# Posted Forever — "Can the people who run the county live in the county?"
8th Annual Datathon for Social Good — reproducible pipeline (Indeed + Zillow + Google Places).

**Setup:** place the dataset files at these relative paths before running:
- `data_zillow/Zillow_part1.csv`, `data_zillow/zillow_part2.csv`, `data_zillow/zillow_part3.csv`
- `data/google maps dataset/google_maps.csv`
- `data/indeed/indeed_job_listings.csv` (full Indeed file)
- `output_city/city_joined.csv` and `output_city/city_tableau.csv` (from the team's city pipeline, for county names + the 128-city Bay Area set)

Run the cells in order. Final Tableau-ready CSVs land in `output_posted_forever/` with `findings_v2.md`.
Caveats: Indeed = online postings since Jan 2024, not civil-service pay tables; missing salaries create selection bias; Zillow 1BR medians = sampled FOR_RENT listings (what a new hire faces); commute shadows are straight-line miles.

## Step 1 — Zillow: 1BR listing rents by city (FOR_RENT rows only, so `price` is monthly rent)

In [ ]:
"""Reproducible build of the v1/v2 rent table (saved Nov 14, 2026 for the Project Link notebook).
Zillow FOR_RENT rows ONLY (homeStatus == 'FOR_RENT' -> price is monthly rent), bedrooms == 1,
0 < price < 20000 sanity band, median by standardized city. City-level aggregates only.
Output: output/zillow_1br_rent_listings_by_city.csv (city, n_listings_1br, med_rent_listing_1br, lat, lon)
"""
import pandas as pd, numpy as np, re
def std_city(s):
    if pd.isna(s): return np.nan
    s = re.sub(r"\s+", " ", str(s).strip().lower()).replace(".", "")
    abbr = {"mt":"mountain","st":"saint","ft":"fort","pt":"point"}
    s = " ".join(abbr.get(p,p) for p in s.split(" ")).title()
    return {"Losaltos":"Los Altos","Mountainview":"Mountain View","Paloalto":"Palo Alto","Sanjose":"San Jose",
            "The Sea Ranch":"Sea Ranch","Eastpaloalto":"East Palo Alto","Sanmateo":"San Mateo","Sanramon":"San Ramon",
            "Walnutcreek":"Walnut Creek","Mountainhouse":"Mountain House"}.get(s, s)
frames = []
for f in ["data_zillow/Zillow_part1.csv","data_zillow/zillow_part2.csv","data_zillow/zillow_part3.csv"]:
    hdr = pd.read_csv(f, nrows=0).columns.tolist()
    keep = [c for c in hdr if c in ["city","homeStatus","bedrooms","price","latitude","longitude"]]
    frames.append(pd.read_csv(f, usecols=keep, low_memory=True))
z = pd.concat(frames, ignore_index=True)
fr = z[z["homeStatus"] == "FOR_RENT"]
fr1 = fr[(fr["bedrooms"] == 1) & (fr["price"] > 0) & (fr["price"] < 20000)].copy()
fr1["city"] = fr1["city"].map(std_city)
g = fr1.groupby("city").agg(n_listings_1br=("price","size"), med_rent_listing_1br=("price","median"),
                            lat=("latitude","median"), lon=("longitude","median")).reset_index()
g.to_csv("output/zillow_1br_rent_listings_by_city.csv", index=False)
print(f"FOR_RENT rows: {len(fr)} | 1BR used: {len(fr1)} | cities: {len(g)}")


## Step 2 — Places: essentials per city (open locations only: grocery, transit, clinics)

In [ ]:
"""Places essentials per city (saved Nov 14, 2026 for reproducibility).
Open locations only (permanently_closed == False); cities restricted to the established
128-city Bay Area tableau set (9 counties; Santa Cruz city excluded upstream).
Output: output/city_essentials.csv
"""
import pandas as pd, numpy as np, re
def std_city(s):
    if pd.isna(s): return np.nan
    s = re.sub(r"\s+", " ", str(s).strip().lower()).replace(".", "")
    abbr = {"mt":"mountain","st":"saint","ft":"fort","pt":"point"}
    s = " ".join(abbr.get(p,p) for p in s.split(" ")).title()
    return {"Losaltos":"Los Altos","Mountainview":"Mountain View","Paloalto":"Palo Alto","Sanjose":"San Jose",
            "The Sea Ranch":"Sea Ranch","Eastpaloalto":"East Palo Alto","Sanmateo":"San Mateo","Sanramon":"San Ramon",
            "Walnutcreek":"Walnut Creek","Mountainhouse":"Mountain House"}.get(s, s)
p = pd.read_csv("data/google maps dataset/google_maps.csv",
                usecols=["category","address","permanently_closed"], low_memory=True)
p = p[p["permanently_closed"] == False].copy()
p["city_raw"] = p["address"].astype(str).str.extract(r"(?:^|,\s*)([^,]+),\s*CA(?:\s+\d{5})?(?:,|$)")
p = p[p["city_raw"].notna()].copy()
p["city"] = p["city_raw"].map(std_city)
def kind(c):
    if c in ("Supermarket","Grocery store"): return "grocery"
    if c in ("Bus stop","Transit station"): return "transit"
    if isinstance(c, str) and "clinic" in c.lower() and "veterinary" not in c.lower(): return "clinic"
    return None
p["kind"] = p["category"].map(kind)
p = p[p["kind"].notna()].copy()
tab = pd.read_csv("output_city/city_tableau.csv")[["city"]]
ess = p.groupby(["city","kind"]).size().unstack(fill_value=0).reset_index()
for c in ["grocery","transit","clinic"]:
    if c not in ess: ess[c] = 0
ess = tab.merge(ess, on="city", how="left").fillna(0)
ess["has_grocery"] = ess["grocery"] > 0; ess["has_transit"] = ess["transit"] > 0; ess["has_clinic"] = ess["clinic"] > 0
ess["has_all_essentials"] = ess["has_grocery"] & ess["has_transit"] & ess["has_clinic"]
ess.to_csv("output/city_essentials.csv", index=False)
print("cities:", len(ess), "| has all essentials:", int(ess["has_all_essentials"].sum()))


## Step 3 — Indeed v2: SB 1162 salary flags, dedupe, persona matching, affordability, commute shadows

In [ ]:
"""
Indeed v2 build — implements the v2 spec (Nov 14, 2026) sections 2-4.
Order of operations (per spec): standardize city -> flag has_posted_salary on RAW rows (SB 1162)
-> parse salary to hourly+annual -> persona match -> dedupe on job_title+company_name+location
-> all medians/counts/rates computed AFTER dedupe (transparency also reported pre-dedupe).
Personas (title-based, spec section 3): Teachers, Nurses, Law Enforcement, Bus/Transit Operators,
Social Workers, Librarians, 911 Dispatchers. County-worker scope = public employer OR persona match
at a public-sector-type employer; the grouped tier 'Public-Sector / Essential Services' pools all.
Sparsity safeguard: persona x city cells with n_with_salary < 10 are flagged cell_reliable = False
and excluded from chart layers; Bay Area-wide persona rows are always emitted.
Outputs (output_posted_forever/): persona_affordability_summary.csv, city_jobs_summary.csv,
commute_shadow_lines.csv, findings_v2.md (overwrites the sample findings when run on full data).
Usage: python3 indeed_v2_build.py [--sample]
"""
import re, sys, os
import numpy as np
import pandas as pd
from math import radians, sin, cos, asin, sqrt

SAMPLE = "--sample" in sys.argv
SRC = "data/indeed/indeed_sample.csv" if SAMPLE else "data/indeed/indeed_job_listings.csv"
os.makedirs("output_posted_forever", exist_ok=True)
WEEKS_PER_MONTH = 52 / 12  # 4.333

def std_city(s):
    if pd.isna(s): return np.nan
    s = re.sub(r"\s+", " ", str(s).strip().lower()).replace(".", "")
    abbr = {"mt":"mountain","st":"saint","ft":"fort","pt":"point"}
    s = " ".join(abbr.get(p,p) for p in s.split(" ")).title()
    return {"Losaltos":"Los Altos","Mountainview":"Mountain View","Paloalto":"Palo Alto","Sanjose":"San Jose",
            "The Sea Ranch":"Sea Ranch","Eastpaloalto":"East Palo Alto","Sanmateo":"San Mateo","Sanramon":"San Ramon",
            "Walnutcreek":"Walnut Creek","Mountainhouse":"Mountain House"}.get(s, s)

def parse_salary(v):
    """-> (hourly, annual, kind). Midpoint of ranges. Daily rates x5x52, weekly x52, monthly x12."""
    if not isinstance(v, str): return (np.nan, np.nan, "")
    s = v.replace(",", "").lower()
    nums = [float(x) for x in re.findall(r"\$(\d+(?:\.\d+)?)", s)]
    if not nums: return (np.nan, np.nan, "")
    val = nums[0] if len(nums) == 1 else (nums[0] + nums[1]) / 2
    kind = "from" if s.startswith("from") else ("up_to" if s.startswith("up to") else "range" if len(nums) > 1 else "single")
    if "hour" in s: return (val, val * 2080, kind)
    if "year" in s or "annual" in s: return (val / 2080, val, kind)
    if "month" in s: return (val * 12 / 2080, val * 12, kind)
    if "week" in s: return (val * 52 / 2080, val * 52, kind)
    if "day" in s: return (val * 260 / 2080, val * 260, kind)
    return (np.nan, np.nan, "")

PUB_COMPANY = re.compile(r"county of|city of|town of|state of california|sheriff|police department|"
                         r"fire department|fire district|unified school district|school district|bart|"
                         r"bay area rapid transit|water district|transit district|port of|superior court|"
                         r"municipal|public works|library district|hospital authority|health system|"
                         r"university of california|california state university|community college", re.I)
PERSONAS = [
    ("911 Dispatchers", re.compile(r"dispatcher|public safety communicator|emergency communications", re.I)),
    ("Law Enforcement", re.compile(r"deputy sheriff|sheriff|correctional officer|detention officer|police officer|patrol|probation officer", re.I)),
    ("Nurses", re.compile(r"\bnurse\b|nursing|registered nurse|\brn\b|\blvn\b|licensed vocational", re.I)),
    ("Teachers", re.compile(r"\bteacher\b|special education|substitute teacher|preschool|\beducator\b", re.I)),
    ("Bus/Transit Operators", re.compile(r"bus operator|transit operator|coach operator|bus driver", re.I)),
    ("Social Workers", re.compile(r"social worker|case manager|human services", re.I)),
    ("Librarians", re.compile(r"librarian|library assistant|library technician|library tech", re.I)),
]
def persona_of(title):
    t = str(title or "")
    for name, pat in PERSONAS:
        if pat.search(t): return name
    return None

# ---- load + standardize (spec 2) ----
rent = pd.read_csv("output/zillow_1br_rent_listings_by_city.csv")
RELIABLE59 = set(rent[rent["n_listings_1br"] >= 30]["city"])
rent_map = dict(zip(rent["city"], rent["med_rent_listing_1br"]))
lat_map = dict(zip(rent["city"], rent["lat"])); lon_map = dict(zip(rent["city"], rent["lon"]))

df = pd.read_csv(SRC, low_memory=True, dtype=str)
n_raw = len(df)
df["city"] = df["location"].fillna("").str.extract(r"^([^,]+),\s*CA")[0].map(std_city)
df = df[df["city"].isin(RELIABLE59)].copy()   # normalize to the 59 reliable cities
# SB 1162 flag BEFORE any filtering/dedupe (spec 2)
parsed = df["salary_formatted"].map(parse_salary)
df["hourly"] = [p[0] for p in parsed]; df["annual"] = [p[1] for p in parsed]; df["salary_kind"] = [p[2] for p in parsed]
df["has_posted_salary"] = df["hourly"].notna()
transparency_raw = 1 - df["has_posted_salary"].mean()   # pre-dedupe, reported alongside
df["is_public"] = df["company_name"].fillna("").str.contains(PUB_COMPANY)
df["persona"] = df["job_title"].map(persona_of)
df["is_county_worker"] = df["is_public"] | df["persona"].notna()

# spot-check sample of parsed salaries (spec 2) — written to findings for hand verification
spot = df[df["has_posted_salary"]][["job_title","company_name","city","salary_formatted","hourly","annual"]].sample(
    n=min(10, int(df["has_posted_salary"].sum())), random_state=3) if df["has_posted_salary"].any() else pd.DataFrame()

# ---- dedupe BEFORE medians/counts/rates (spec 2) ----
g = df.groupby(["job_title","company_name","city"]).agg(
    n_postings=("jobid","size"),
    n_with_salary=("has_posted_salary","sum"),
    hourly_median=("hourly","median"), annual_median=("annual","median"),
    is_public=("is_public","max"), is_county_worker=("is_county_worker","max"),
    persona=("persona","first"),
).reset_index()
g["pct_missing_salary"] = (100 * (1 - g["n_with_salary"] / g["n_postings"])).round(1)
g["rent_1br"] = g["city"].map(rent_map)
g["rent_burden_pct"] = (g["rent_1br"] * 12 / g["annual_median"] * 100).round(1)
g["weekly_hours_for_rent"] = (g["rent_1br"] / (g["hourly_median"] * WEEKS_PER_MONTH)).round(1)
g["affordable_flag"] = g["rent_burden_pct"] <= 30

# ---- sparsity report FIRST (spec 3) ----
sparsity = []
for name, _ in PERSONAS:
    sub = df[df["persona"] == name]
    sparsity.append((name, len(sub), int(sub["has_posted_salary"].sum())))
pool = df[df["is_county_worker"]]

# ---- output 1: persona affordability (role x city + Bay Area rows) ----
cw = g[g["is_county_worker"] & g["persona"].notna()].copy()
rows = []
for (persona, city), sub in cw.groupby(["persona","city"]):
    sal = sub[sub["hourly_median"].notna()]
    rows.append({"persona": persona, "city": city, "n_postings": int(sub["n_postings"].sum()),
        "n_role_groups": len(sub), "n_with_salary": int(sub["n_with_salary"].sum()),
        "pct_missing_salary": round(100 * (1 - sub["n_with_salary"].sum() / max(sub["n_postings"].sum(),1)), 1),
        "median_hourly": round(sub["hourly_median"].median(), 2) if len(sal) else np.nan,
        "median_annual": round(sub["annual_median"].median(), 0) if len(sal) else np.nan,
        "rent_1br": rent_map.get(city), "rent_burden_pct": round(sub["rent_burden_pct"].median(), 1) if len(sal) else np.nan,
        "weekly_hours_for_rent": round(sub["weekly_hours_for_rent"].median(), 1) if len(sal) else np.nan,
        "affordable_flag": bool(sub["rent_burden_pct"].median() <= 30) if len(sal) else np.nan,
        "cell_reliable": bool(sub["n_with_salary"].sum() >= 10)})
for persona, sub in cw.groupby("persona"):
    sal = sub[sub["hourly_median"].notna()]
    rows.append({"persona": persona, "city": "BAY AREA (all 59 cities)", "n_postings": int(sub["n_postings"].sum()),
        "n_role_groups": len(sub), "n_with_salary": int(sub["n_with_salary"].sum()),
        "pct_missing_salary": round(100 * (1 - sub["n_with_salary"].sum() / max(sub["n_postings"].sum(),1)), 1),
        "median_hourly": round(sub["hourly_median"].median(), 2) if len(sal) else np.nan,
        "median_annual": round(sub["annual_median"].median(), 0) if len(sal) else np.nan,
        "rent_1br": np.nan, "rent_burden_pct": np.nan, "weekly_hours_for_rent": np.nan,
        "affordable_flag": np.nan, "cell_reliable": bool(sub["n_with_salary"].sum() >= 10)})
# grouped tier
sub = g[g["is_county_worker"]]
sal = sub[sub["hourly_median"].notna()]
rows.append({"persona": "Public-Sector / Essential Services (all)", "city": "BAY AREA (all 59 cities)",
    "n_postings": int(sub["n_postings"].sum()), "n_role_groups": len(sub), "n_with_salary": int(sub["n_with_salary"].sum()),
    "pct_missing_salary": round(100 * (1 - sub["n_with_salary"].sum() / max(sub["n_postings"].sum(),1)), 1),
    "median_hourly": round(sub["hourly_median"].median(), 2) if len(sal) else np.nan,
    "median_annual": round(sub["annual_median"].median(), 0) if len(sal) else np.nan,
    "rent_1br": np.nan, "rent_burden_pct": np.nan, "weekly_hours_for_rent": np.nan,
    "affordable_flag": np.nan, "cell_reliable": True})
pd.DataFrame(rows).to_csv("output_posted_forever/persona_affordability_summary.csv", index=False)

# ---- output 2: city jobs summary (59-city update) ----
cj_rows = []
for city in sorted(RELIABLE59):
    allg = g[g["city"] == city]; cwg = allg[allg["is_county_worker"]]
    def share(s):
        s2 = s[s["hourly_median"].notna()]
        return round(100 * s2["affordable_flag"].mean(), 1) if len(s2) else np.nan
    raw_city = df[df["city"] == city]
    cj_rows.append({"city": city, "rent_1br": rent_map.get(city),
        "n_postings_all": int(raw_city.shape[0]), "n_role_groups_all": len(allg),
        "share_all_jobs_affordable_pct": share(allg), "share_county_worker_affordable_pct": share(cwg),
        "pct_postings_missing_salary": round(100 * (1 - raw_city["has_posted_salary"].mean()), 1) if len(raw_city) else np.nan,
        "pct_county_worker_missing_salary": round(100 * (1 - raw_city[raw_city["is_county_worker"]]["has_posted_salary"].mean()), 1) if raw_city["is_county_worker"].any() else np.nan})
pd.DataFrame(cj_rows).to_csv("output_posted_forever/city_jobs_summary.csv", index=False)

# ---- output 3: commute shadow lines (persona x hub) ----
def hav(a,b,c,d):
    R=3958.8; p1,p2=radians(a),radians(c); dp,dl=radians(c-a),radians(d-b)
    h=sin(dp/2)**2+cos(p1)*cos(p2)*sin(dl/2)**2; return 2*R*asin(sqrt(h))
HUBS = ["San Francisco","San Jose","Oakland","Redwood City","San Rafael","Santa Rosa","Napa","Fairfield","Martinez"]
aff_rent = rent[rent["n_listings_1br"] >= 10]
lines = []
persona_pay = cw.groupby("persona")["hourly_median"].median()
for persona, pay in persona_pay.items():
    if pd.isna(pay): continue
    budget = pay * 52
    ok = aff_rent[aff_rent["med_rent_listing_1br"] <= budget]
    for hub in HUBS:
        if hub not in lat_map: continue
        if len(ok) == 0:
            lines.append({"persona": persona, "work_city": hub, "work_lat": lat_map[hub], "work_lon": lon_map[hub],
                          "home_city": "None in 59 cities", "home_lat": np.nan, "home_lon": np.nan, "miles": np.nan,
                          "persona_median_hourly": round(pay,2)})
        else:
            d = ok.apply(lambda r: hav(lat_map[hub], lon_map[hub], r["lat"], r["lon"]), axis=1)
            i = d.idxmin()
            lines.append({"persona": persona, "work_city": hub, "work_lat": lat_map[hub], "work_lon": lon_map[hub],
                          "home_city": ok.loc[i,"city"], "home_lat": ok.loc[i,"lat"], "home_lon": ok.loc[i,"lon"],
                          "miles": round(float(d[i]),1), "persona_median_hourly": round(pay,2)})
pd.DataFrame(lines).to_csv("output_posted_forever/commute_shadow_lines.csv", index=False)

# ---- findings ----
L = [f"# Posted Forever v2 findings — {'SAMPLE smoke test' if SAMPLE else 'FULL FILE'} — Nov 14, 2026\n",
     f"Indeed rows in: {n_raw} | located in the 59 reliable cities: {len(df)} | deduped role groups: {len(g)}",
     f"SB 1162 transparency: {100*transparency_raw:.1f}% of located postings omit salary (pre-dedupe); "
     f"{100*(1-g['n_with_salary'].sum()/g['n_postings'].sum()):.1f}% post-dedupe (posting-weighted).",
     "\n## Persona sparsity (spec 3 — read before charts)"]
for name, n, ns in sparsity:
    L.append(f"- {name}: {n} matched postings in 59 cities, {ns} with posted salary")
L.append(f"- County-worker pool (public employer or persona): {len(pool)} postings, {int(pool['has_posted_salary'].sum())} with salary")
L.append("\n## Salary parse spot-check (hand-verify these)")
if len(spot):
    for _, r in spot.iterrows():
        L.append(f"- '{r['salary_formatted']}' -> ${r['hourly']:.2f}/hr (${r['annual']:,.0f}/yr) — {r['job_title']} @ {r['company_name']} ({r['city']})")
L.append("\n## Caveats (spec 5)")
L.append("Indeed = online postings since Jan 2024, not civil-service pay tables; missing salaries create selection bias; "
         "Zillow 1BR medians = sampled FOR_RENT listings (what a new hire faces), not family units; "
         "commute shadows are straight-line miles and understate real Bay Area travel time; "
         "posted pay excludes benefits, pensions, overtime and step increases.")
open("output_posted_forever/findings_v2.md","w").write("\n".join(L))
print("\n".join(L))


## Outputs
- `output_posted_forever/persona_affordability_summary.csv` — role x city + Bay Area persona rows
- `output_posted_forever/city_jobs_summary.csv` — 59-city share-affordable + salary-omission rates
- `output_posted_forever/commute_shadow_lines.csv` — persona x hub -> nearest affordable city (MAKELINE-ready)
- `output_posted_forever/findings_v2.md` — sparsity report, transparency rates, salary spot-checks, caveats

## Step 4 — Tableau v2 prep (augmented rent table, commute-shadow points, persona matrix)
Run after step 3. Source: `tableau_v2_prep.py`.

In [ ]:
"""Tableau v2 prep — Posted Forever. Builds, from the full Indeed run:
1. output_posted_forever/city_rent_tableau_v2.csv
   = city_rent_tableau.csv (all original columns, same rows) + per-city Indeed stats:
     median posted hourly pay (all postings, deduped), n postings, burden at that pay,
     % of local postings affordable, % county-worker postings affordable, % missing salary.
2. output_posted_forever/commute_shadow_points.csv  (long format for Tableau MAKELINE/line layer)
3. output_posted_forever/persona_city_matrix.csv     (7 personas x 59 cities at Bay Area persona median pay)
4. output_posted_forever/persona_benchmarks.csv      (persona median wage + # of 59 cities affordable)
Cleaning functions replicate indeed_v2_build.py exactly (same extraction, parse, dedupe).
"""
import re
import numpy as np
import pandas as pd

OUT = "output_posted_forever"
WEEKS_PER_MONTH = 52 / 12

def std_city(s):
    if pd.isna(s): return np.nan
    s = re.sub(r"\s+", " ", str(s).strip().lower()).replace(".", "")
    abbr = {"mt": "mountain", "st": "saint", "ft": "fort", "pt": "point"}
    s = " ".join(abbr.get(p, p) for p in s.split(" ")).title()
    return {"Losaltos": "Los Altos", "Mountainview": "Mountain View", "Paloalto": "Palo Alto",
            "Sanjose": "San Jose", "The Sea Ranch": "Sea Ranch", "Eastpaloalto": "East Palo Alto",
            "Sanmateo": "San Mateo", "Sanramon": "San Ramon", "Walnutcreek": "Walnut Creek",
            "Mountainhouse": "Mountain House"}.get(s, s)

def parse_salary(v):
    if not isinstance(v, str): return (np.nan, np.nan)
    s = v.replace(",", "").lower()
    nums = [float(x) for x in re.findall(r"\$(\d+(?:\.\d+)?)", s)]
    if not nums: return (np.nan, np.nan)
    val = nums[0] if len(nums) == 1 else (nums[0] + nums[1]) / 2
    if "hour" in s: return (val, val * 2080)
    if "year" in s or "annual" in s: return (val / 2080, val)
    if "month" in s: return (val * 12 / 2080, val * 12)
    if "week" in s: return (val * 52 / 2080, val * 52)
    if "day" in s: return (val * 260 / 2080, val * 260)
    return (np.nan, np.nan)

rent = pd.read_csv("output/zillow_1br_rent_listings_by_city.csv")
RELIABLE59 = set(rent[rent["n_listings_1br"] >= 30]["city"])

df = pd.read_csv("data/indeed/indeed_job_listings.csv", low_memory=True, dtype=str,
                 usecols=["jobid", "job_title", "company_name", "location", "salary_formatted"])
df["city"] = df["location"].fillna("").str.extract(r"([^,]+),\s*CA")[0].map(std_city)
df = df[df["city"].isin(RELIABLE59)].copy()
parsed = df["salary_formatted"].map(parse_salary)
df["hourly"] = [p[0] for p in parsed]
g = df.groupby(["job_title", "company_name", "city"]).agg(
    n_postings=("jobid", "size"), hourly_median=("hourly", "median"),
    n_with_salary=("hourly", lambda s: int(s.notna().sum()))).reset_index()

city_pay = g.groupby("city").agg(
    median_posted_hourly_all=("hourly_median", "median"),
    n_postings_all=("n_postings", "sum"),
    n_with_salary_all=("n_with_salary", "sum")).reset_index()
city_pay["median_posted_hourly_all"] = city_pay["median_posted_hourly_all"].round(2)
city_pay["pay_reliable"] = city_pay["n_with_salary_all"] >= 10

# ---- 1. augmented rent table ----
tab = pd.read_csv(f"{OUT}/city_rent_tableau.csv")
jobs = pd.read_csv(f"{OUT}/city_jobs_summary.csv")[
    ["city", "share_all_jobs_affordable_pct", "share_county_worker_affordable_pct",
     "pct_postings_missing_salary", "pct_county_worker_missing_salary"]]
m = tab.merge(city_pay, on="city", how="left").merge(jobs, on="city", how="left")
m["burden_at_median_posted_pay_pct"] = (
    m["median_rent_1br_listings"] * 12 / (m["median_posted_hourly_all"] * 2080) * 100).round(1)
m["weekly_hours_at_median_posted_pay"] = (
    m["median_rent_1br_listings"] / (m["median_posted_hourly_all"] * WEEKS_PER_MONTH)).round(1)
m.to_csv(f"{OUT}/city_rent_tableau_v2.csv", index=False)
print(f"city_rent_tableau_v2.csv: {len(m)} rows; cities with posted-pay medians: "
      f"{int(m['median_posted_hourly_all'].notna().sum())}")

# ---- 2. commute shadow, long format ----
lines = pd.read_csv(f"{OUT}/commute_shadow_lines.csv")
valid = lines[lines["home_lat"].notna()].copy()
pts = []
for _, r in valid.iterrows():
    lid = f"{r['persona']} | {r['work_city']}"
    for order, lat, lon, kind in [(1, r["work_lat"], r["work_lon"], "Work hub"),
                                  (2, r["home_lat"], r["home_lon"], "Nearest affordable city")]:
        pts.append({"line_id": lid, "point_order": order, "lat": lat, "lon": lon,
                    "persona": r["persona"], "work_city": r["work_city"],
                    "home_city": r["home_city"], "point_kind": kind,
                    "miles": r["miles"], "persona_median_hourly": r["persona_median_hourly"]})
pts = pd.DataFrame(pts)
pts.to_csv(f"{OUT}/commute_shadow_points.csv", index=False)
n_none = int(lines["home_lat"].isna().sum())
print(f"commute_shadow_points.csv: {len(pts)} points / {len(valid)} lines; "
      f"{n_none} persona-hub combos have NO affordable city in the 59")

# ---- 3 + 4. persona matrix + benchmarks ----
pers = pd.read_csv(f"{OUT}/persona_affordability_summary.csv")
ba = pers[pers["city"].str.contains("BAY AREA") &
          (pers["persona"] != "Public-Sector / Essential Services (all)")].copy()
rel = tab[tab["reliable"] == True][["city", "county", "median_rent_1br_listings", "pay_needed_per_hr"]]
rows, bench = [], []
for _, p in ba.iterrows():
    annual = p["median_hourly"] * 2080
    afford = int((rel["pay_needed_per_hr"] <= p["median_hourly"]).sum())
    bench.append({"persona": p["persona"], "median_hourly": p["median_hourly"],
                  "median_annual": p["median_annual"], "n_with_salary": int(p["n_with_salary"]),
                  "pct_missing_salary": p["pct_missing_salary"],
                  "cities_affordable_of_59": afford})
    for _, c in rel.iterrows():
        burden = c["median_rent_1br_listings"] * 12 / annual * 100
        rows.append({"persona": p["persona"], "city": c["city"], "county": c["county"],
                     "persona_median_hourly": p["median_hourly"],
                     "median_rent_1br": c["median_rent_1br_listings"],
                     "rent_burden_pct": round(burden, 1),
                     "weekly_hours_for_rent": round(
                         c["median_rent_1br_listings"] / (p["median_hourly"] * WEEKS_PER_MONTH), 1),
                     "affordable_flag": burden <= 30})
pd.DataFrame(rows).to_csv(f"{OUT}/persona_city_matrix.csv", index=False)
bench = pd.DataFrame(bench).sort_values("median_hourly")
bench.to_csv(f"{OUT}/persona_benchmarks.csv", index=False)
print(f"persona_city_matrix.csv: {len(rows)} rows (7 personas x 59 cities)")
print(bench.to_string(index=False))
